## Testing

In [2]:
# Install packages if you haven't already:
# using Pkg
# Pkg.add("PowerModels")
# Pkg.add("Ipopt")
# Pkg.add("JSON")
# Pkg.add("JuMP")

using PowerModels
using Ipopt
using JSON
using JuMP
using Random

# Set a random seed so results are reproducible
Random.seed!(42)

# Suppress PowerModels' internal logging (it's very verbose otherwise)
PowerModels.silence()

[info | PowerModels]: Suppressing information and warning messages for the rest of this session.  Use the Memento package for more fine-grained control of logging.


In [3]:
# --- Configuration ---
N_SAMPLES = 300                          # How many data points to generate
NOISE_STD = 0.1                          # 10% standard deviation
CASE_FILE = "pglib_opf_case57_ieee.m"    # Path to the case file
OUTPUT_DIR = "pf_dataset"                # Folder to save JSON files

# Create output directory if it doesn't exist
mkpath(OUTPUT_DIR)

"pf_dataset"

In [4]:
# --- Load the base case data ---
base_data = PowerModels.parse_file(CASE_FILE)

# Let's peek at what we have
println("Number of buses: ", length(base_data["bus"]))
println("Number of generators: ", length(base_data["gen"]))
println("Number of loads: ", length(base_data["load"]))
println("Number of branches: ", length(base_data["branch"]))

Number of buses: 57
Number of generators: 7
Number of loads: 42
Number of branches: 80


In [5]:
"""
    perturb_loads!(data, noise_std)

Apply Gaussian noise to all active loads in the network.
Each load gets its own random multiplier applied to both pd and qd,
preserving the power factor.

Returns a dictionary recording what multipliers were applied.
"""
function perturb_loads!(data, noise_std)
    multipliers = Dict{String, Float64}()

    for (load_id, load) in data["load"]
        # Skip loads that are turned off
        if load["status"] == 0
            continue
        end

        # Skip zero loads (multiplying zero by anything stays zero)
        if load["pd"] == 0.0 && load["qd"] == 0.0
            multipliers[load_id] = 1.0
            continue
        end

        # Draw a random multiplier: mean=1, std=noise_std
        # So for noise_std=0.1, we get values like 0.85, 1.12, 0.97, etc.
        multiplier = 1.0 + noise_std * randn()

        # Apply to both pd and qd (preserves power factor)
        load["pd"] *= multiplier
        load["qd"] *= multiplier

        # Record what we did
        multipliers[load_id] = multiplier
    end

    return multipliers
end

perturb_loads!

In [6]:
"""
    solve_pf_with_iterations(data)

Solve power flow by building the JuMP model manually,
so we can access Ipopt's iteration count afterward.
"""
function solve_pf_with_iterations(data)
    # Configure Ipopt
    solver = optimizer_with_attributes(
        Ipopt.Optimizer,
        "print_level" => 0,
        "max_iter" => 100,
        "tol" => 1e-8,
    )

    # Use PowerModels' instantiate_model to build the JuMP model
    # without immediately solving it
    pm = PowerModels.instantiate_model(
        data,
        ACPPowerModel,
        PowerModels.build_pf    # This sets up the power flow equations
    )

    # Now optimize (solve) the model
    JuMP.optimize!(pm.model)

    # Extract iteration count from Ipopt via MOI attributes
    iterations = -1
    try
        # Ipopt exposes the barrier iteration count
        iterations = MOI.get(pm.model, MOI.BarrierIterations())
    catch
        # If that doesn't work, try the simplex iterations attribute
        try
            iterations = MOI.get(pm.model, MOI.SimplexIterations())
        catch
            # Could not retrieve iterations
            iterations = -1
        end
    end

    # Use PowerModels to build the result dictionary from the solved model
    result = PowerModels.build_result(pm, Float64)

    # Get solve time
    solve_time = JuMP.solve_time(pm.model)
    result["solve_time"] = solve_time

    return result, iterations
end

solve_pf_with_iterations

In [7]:
"""
    save_sample(filepath, sample_data)

Write a single sample to a JSON file.
"""
function save_sample(filepath, sample_data)
    open(filepath, "w") do f
        JSON.print(f, sample_data, 4)  # 4 = indent spaces for readability
    end
end

# --- Main generation loop ---
println("Starting dataset generation...")
println("Samples to generate: $N_SAMPLES")
println("Noise level: $(NOISE_STD * 100)%")
println()

n_converged = 0
n_failed = 0

for i in 1:N_SAMPLES
    # Deep copy so we don't modify the original base case
    data = deepcopy(base_data)

    # Perturb the loads
    multipliers = perturb_loads!(data, NOISE_STD)

    # Solve power flow
    result, iterations = solve_pf_with_iterations(data)

    # Check if the solver converged
    term_status = result["termination_status"]
    converged = (term_status == LOCALLY_SOLVED || term_status == OPTIMAL)

    if !converged
        n_failed += 1
        println("  Sample $i: FAILED ($term_status)")
        continue
    end

    n_converged += 1

    # --- Build the output data structure ---
    sample_data = Dict(
        "sample_id" => i,
        "termination_status" => string(term_status),
        "solve_time_seconds" => result["solve_time"],
        "solver_iterations" => iterations,

        # INPUTS: what we fed into the solver
        "inputs" => Dict(
            "loads" => Dict(
                load_id => Dict(
                    "pd" => data["load"][load_id]["pd"],
                    "qd" => data["load"][load_id]["qd"],
                    "load_bus" => data["load"][load_id]["load_bus"],
                    "multiplier_applied" => multipliers[load_id]
                )
                for load_id in keys(data["load"])
            ),
            "generators" => Dict(
                gen_id => Dict(
                    "pg" => data["gen"][gen_id]["pg"],
                    "vm" => data["gen"][gen_id]["vg"],
                    "gen_bus" => data["gen"][gen_id]["gen_bus"]
                )
                for gen_id in keys(data["gen"])
            )
        ),

        # OUTPUTS: what the solver computed
        "outputs" => Dict(
            "bus" => Dict(
                bus_id => Dict(
                    "vm" => bus_sol["vm"],
                    "va" => bus_sol["va"]
                )
                for (bus_id, bus_sol) in result["solution"]["bus"]
            ),
            "gen" => Dict(
                gen_id => Dict(
                    "pg" => gen_sol["pg"],
                    "qg" => gen_sol["qg"]
                )
                for (gen_id, gen_sol) in result["solution"]["gen"]
            )
        )
    )

    # Save to file
    filename = "sample_$(lpad(i, 4, '0')).json"
    filepath = joinpath(OUTPUT_DIR, filename)
    save_sample(filepath, sample_data)

    # Progress update every 50 samples
    if i % 50 == 0
        println("  Completed $i / $N_SAMPLES samples " *
                "($n_converged converged, $n_failed failed)")
    end
end

# --- Summary ---
println()
println("=" ^ 50)
println("Dataset generation complete!")
println("  Total samples attempted: $N_SAMPLES")
println("  Converged: $n_converged")
println("  Failed: $n_failed")
println("  Convergence rate: $(round(n_converged/N_SAMPLES*100, digits=1))%")
println("  Output directory: $OUTPUT_DIR")
println("=" ^ 50)

Starting dataset generation...
Samples to generate: 300
Noise level: 10.0%



NoOptimizer: NoOptimizer()

In [8]:
# Just test that the base case solves at all
data = PowerModels.parse_file("pglib_opf_case57_ieee.m")
result = PowerModels.solve_pf(data, ACPPowerModel, Ipopt.Optimizer)
println(result["termination_status"])
println(keys(result["solution"]))


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:      880
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:     2526

Total number of variables............................:      128
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:      128
Total number of inequality c

In [9]:
result = PowerModels.solve_pf(data, ACPPowerModel, Ipopt.Optimizer)
println(typeof(result["termination_status"]))
println(result["termination_status"] == MOI.LOCALLY_SOLVED)

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:      880
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:     2526

Total number of variables............................:      128
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:      128
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 5.79e+00 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [13]:
using JuMP

pm = PowerModels.instantiate_model(data, ACPPowerModel, PowerModels.build_pf)
JuMP.optimize!(pm.model)

# Try getting iteration count
println("Barrier iterations: ", MOI.get(pm.model, MOI.BarrierIterations()))

NoOptimizer: NoOptimizer()

In [14]:
println("Solve time: ", JuMP.solve_time(pm.model))
println("Term status: ", JuMP.termination_status(pm.model))

┌ Warning: The model has been modified since the last call to `optimize!` (or `optimize!` has not been called yet). If you are iteratively querying solution information and modifying a model, query all the results first, then modify the model.
└ @ JuMP /home/vaishnavi/.julia/packages/JuMP/EHXNP/src/optimizer_interface.jl:1212


OptimizeNotCalled: OptimizeNotCalled()

In [15]:
using JuMP
using Ipopt

pm = PowerModels.instantiate_model(data, ACPPowerModel, PowerModels.build_pf)

# This is the missing step — attach the solver to the model
JuMP.set_optimizer(pm.model, Ipopt.Optimizer)
JuMP.set_attribute(pm.model, "print_level", 0)

# Now solve
JuMP.optimize!(pm.model)

# Check status
println("Termination: ", JuMP.termination_status(pm.model))
println("Solve time: ", JuMP.solve_time(pm.model))

# Try barrier iterations
try
    println("Barrier iterations: ", MOI.get(pm.model, MOI.BarrierIterations()))
catch e
    println("BarrierIterations not available: ", e)
end

Termination: LOCALLY_SOLVED
Solve time: 0.010210037231445312
Barrier iterations: 4


In [19]:
using Pkg
Pkg.add("Suppressor")

   Resolving package versions...
   Installed Suppressor ─ v0.2.8
    Updating `~/Desktop/Projects/18.337/18337-final-project/Project.toml`
  [fd094767] + Suppressor v0.2.8
    Updating `~/Desktop/Projects/18.337/18337-final-project/Manifest.toml`
  [fd094767] + Suppressor v0.2.8
Precompiling project...
  ✓ Suppressor
  1 dependency successfully precompiled in 1 seconds. 72 already precompiled.


In [20]:
using Suppressor

data = PowerModels.parse_file("pglib_opf_case57_ieee.m")

solver = optimizer_with_attributes(
    Ipopt.Optimizer,
    "print_level" => 5,
)

output = @capture_out begin
    result = PowerModels.solve_pf(data, ACPPowerModel, solver)
end

# Find the iteration line
for line in split(output, "\n")
    if contains(line, "Number of Iterations")
        println("Found it: ", line)
        iterations = parse(Int, strip(split(line, ":")[end]))
        println("Iterations: ", iterations)
    end
end

Found it: Number of Iterations....: 4
Iterations: 4


## Updated code

In [24]:
using PowerModels
using Ipopt
using JSON
using JuMP
using Random
using Suppressor

# Set random seed for reproducibility
Random.seed!(42)

# Suppress PowerModels logging
PowerModels.silence()

# --- Configuration ---
N_SAMPLES = 300
NOISE_STD = 0.1
CASE_FILE = "pglib_opf_case57_ieee.m"
OUTPUT_DIR = "pf_dataset"

mkpath(OUTPUT_DIR)

# --- Load base case ---
base_data = PowerModels.parse_file(CASE_FILE)
println("Loaded case: $(length(base_data["bus"])) buses, " *
        "$(length(base_data["gen"])) generators, " *
        "$(length(base_data["load"])) loads, " *
        "$(length(base_data["branch"])) branches")

# --- Helper: Perturb loads ---
function perturb_loads!(data, noise_std)
    multipliers = Dict{String, Float64}()

    for (load_id, load) in data["load"]
        if load["status"] == 0
            continue
        end

        if load["pd"] == 0.0 && load["qd"] == 0.0
            multipliers[load_id] = 1.0
            continue
        end

        multiplier = 1.0 + noise_std * randn()
        load["pd"] *= multiplier
        load["qd"] *= multiplier
        multipliers[load_id] = multiplier
    end

    return multipliers
end

function solve_pf_with_iterations(data)
    solver = optimizer_with_attributes(
        Ipopt.Optimizer,
        "print_level" => 5,    # Need this on so Ipopt prints iterations
        "max_iter" => 100,
        "tol" => 1e-8,
    )

    # Solve and capture Ipopt's printed output
    local result
    output = @capture_out begin
        result = PowerModels.solve_pf(data, ACPPowerModel, solver)
    end

    # Parse iteration count from output
    iterations = -1
    for line in split(output, "\n")
        if contains(line, "Number of Iterations")
            iterations = parse(Int, strip(split(line, ":")[end]))
            break
        end
    end

    return result, iterations
end

# --- Helper: Save sample ---
function save_sample(filepath, sample_data)
    open(filepath, "w") do f
        JSON.print(f, sample_data, 4)
    end
end

# --- Main generation loop ---
println("\nStarting dataset generation...")
println("  Samples: $N_SAMPLES")
println("  Noise: $(NOISE_STD * 100)%")
println()

n_converged = 0
n_failed = 0

for i in 1:N_SAMPLES
    # Fresh copy of base case
    data = deepcopy(base_data)

    # Perturb loads
    multipliers = perturb_loads!(data, NOISE_STD)

    # Solve power flow
    result, iterations = solve_pf_with_iterations(data)

    # Check convergence
    term_status = result["termination_status"]
    converged = (term_status == MOI.LOCALLY_SOLVED || term_status == MOI.OPTIMAL)

    if !converged
        n_failed += 1
        println("  Sample $i: FAILED ($term_status)")
        continue
    end

    n_converged += 1

    # Build output structure
    sample_data = Dict(
        "sample_id" => i,
        "termination_status" => string(term_status),
        "solve_time_seconds" => result["solve_time"],
        "solver_iterations" => iterations,

        "data" => data,
        "result" => result,
        Dict(
            "loads" => Dict(
                load_id => Dict(
                    "pd" => data["load"][load_id]["pd"],
                    "qd" => data["load"][load_id]["qd"],
                    "load_bus" => data["load"][load_id]["load_bus"],
                    "multiplier_applied" => multipliers[load_id]
                )
                for load_id in keys(data["load"])
            ),
            "generators" => Dict(
                gen_id => Dict(
                    "pg" => data["gen"][gen_id]["pg"],
                    "vm" => data["gen"][gen_id]["vg"],
                    "gen_bus" => data["gen"][gen_id]["gen_bus"]
                )
                for gen_id in keys(data["gen"])
            )
        ),

        "outputs" => Dict(
            "bus" => Dict(
                bus_id => Dict(
                    "vm" => bus_sol["vm"],
                    "va" => bus_sol["va"]
                )
                for (bus_id, bus_sol) in result["solution"]["bus"]
            ),
            "gen" => Dict(
                gen_id => Dict(
                    "pg" => gen_sol["pg"],
                    "qg" => gen_sol["qg"]
                )
                for (gen_id, gen_sol) in result["solution"]["gen"]
            )
        )
    )

    # Save
    filename = "sample_$(lpad(i, 4, '0')).json"
    filepath = joinpath(OUTPUT_DIR, filename)
    save_sample(filepath, sample_data)

    # Progress
    if i % 50 == 0
        println("  Completed $i / $N_SAMPLES " *
                "($n_converged converged, $n_failed failed)")
    end
end

# --- Summary ---
println()
println("=" ^ 50)
println("Dataset generation complete!")
println("  Total attempted: $N_SAMPLES")
println("  Converged: $n_converged")
println("  Failed: $n_failed")
println("  Convergence rate: $(round(n_converged/N_SAMPLES*100, digits=1))%")
println("  Output directory: $OUTPUT_DIR")
println("=" ^ 50)

Loaded case: 57 buses, 7 generators, 42 loads, 80 branches

Starting dataset generation...
  Samples: 300
  Noise: 10.0%

  Completed 50 / 300 (50 converged, 0 failed)
  Completed 100 / 300 (100 converged, 0 failed)
  Completed 150 / 300 (150 converged, 0 failed)
  Completed 200 / 300 (200 converged, 0 failed)
  Completed 250 / 300 (250 converged, 0 failed)
  Completed 300 / 300 (300 converged, 0 failed)

Dataset generation complete!
  Total attempted: 300
  Converged: 300
  Failed: 0
  Convergence rate: 100.0%
  Output directory: pf_dataset


## Adding in more perturbations

In [26]:
using PowerModels
using Ipopt
using JSON
using JuMP
using Random
using Suppressor

# Suppress PowerModels logging
PowerModels.silence()

# Set random seed for reproducibility
Random.seed!(42)

# --- Configuration ---
N_SAMPLES = 1000
LOAD_NOISE_STD = 0.1                     # 10% noise on loads
VOLTAGE_NOISE_STD = 0.02                 # 2% noise on voltage setpoints
VM_MIN = 0.9                             # Minimum allowable voltage setpoint
VM_MAX = 1.1                             # Maximum allowable voltage setpoint
CASE_FILE = "pglib_opf_case57_ieee.m"
OUTPUT_DIR = "pf_dataset"

mkpath(OUTPUT_DIR)

# --- Load base case ---
base_data = PowerModels.parse_file(CASE_FILE)
println("Loaded case: $(length(base_data["bus"])) buses, " *
        "$(length(base_data["gen"])) generators, " *
        "$(length(base_data["load"])) loads, " *
        "$(length(base_data["branch"])) branches")

# --- Helper: Apply flat start ---
"""
    apply_flat_start!(data)

Set all bus voltage magnitudes to 1.0 and angles to 0.0.
This is the initial guess for the solver, not the constraints.
PV buses will still be constrained to their generator's vg setpoint.
"""
function apply_flat_start!(data)
    for (bus_id, bus) in data["bus"]
        bus["vm"] = 1.0
        bus["va"] = 0.0
    end
end

# --- Helper: Perturb loads ---
"""
    perturb_loads!(data, noise_std)

Apply Gaussian noise to all active loads. Same multiplier
for pd and qd (preserves power factor).
"""
function perturb_loads!(data, noise_std)
    multipliers = Dict{String, Float64}()

    for (load_id, load) in data["load"]
        if load["status"] == 0
            continue
        end

        if load["pd"] == 0.0 && load["qd"] == 0.0
            multipliers[load_id] = 1.0
            continue
        end

        multiplier = 1.0 + noise_std * randn()
        load["pd"] *= multiplier
        load["qd"] *= multiplier
        multipliers[load_id] = multiplier
    end

    return multipliers
end

# --- Helper: Perturb generator voltage setpoints ---
"""
    perturb_voltages!(data, noise_std, vm_min, vm_max)

Apply Gaussian noise to generator voltage setpoints (vg).
Clamps to [vm_min, vm_max] to keep things physically reasonable.
Returns a record of original and perturbed values.
"""
function perturb_voltages!(data, noise_std, vm_min, vm_max)
    voltage_perturbations = Dict{String, Dict{String, Float64}}()

    for (gen_id, gen) in data["gen"]
        # Skip inactive generators
        if gen["gen_status"] == 0
            continue
        end

        original_vg = gen["vg"]

        # Add noise: e.g., 1.04 * (1 + 0.02 * randn()) → ~1.04 ± 0.02
        perturbed_vg = original_vg * (1.0 + noise_std * randn())

        # Clamp to physical limits
        perturbed_vg = clamp(perturbed_vg, vm_min, vm_max)

        gen["vg"] = perturbed_vg

        voltage_perturbations[gen_id] = Dict(
            "original_vg" => original_vg,
            "perturbed_vg" => perturbed_vg,
            "gen_bus" => gen["gen_bus"]
        )
    end

    return voltage_perturbations
end

# --- Helper: Solve power flow with iteration count ---
function solve_pf_with_iterations(data)
    solver = optimizer_with_attributes(
        Ipopt.Optimizer,
        "print_level" => 5,
        "max_iter" => 100,
        "tol" => 1e-8,
    )

    local result
    output = @capture_out begin
        result = PowerModels.solve_pf(data, ACPPowerModel, solver)
    end

    # Parse iteration count from Ipopt output
    iterations = -1
    for line in split(output, "\n")
        if contains(line, "Number of Iterations")
            iterations = parse(Int, strip(split(line, ":")[end]))
            break
        end
    end

    return result, iterations
end

# --- Helper: Save sample ---
function save_sample(filepath, sample_data)
    open(filepath, "w") do f
        JSON.print(f, sample_data, 4)
    end
end

# --- Main generation loop ---
println("\nStarting dataset generation...")
println("  Samples: $N_SAMPLES")
println("  Load noise: $(LOAD_NOISE_STD * 100)%")
println("  Voltage noise: $(VOLTAGE_NOISE_STD * 100)%")
println("  Voltage clamp: [$VM_MIN, $VM_MAX] p.u.")
println("  Flat start: enabled")
println()

n_converged = 0
n_failed = 0

for i in 1:N_SAMPLES
    # Fresh copy of base case
    data = deepcopy(base_data)

    # Step 1: Perturb loads
    load_multipliers = perturb_loads!(data, LOAD_NOISE_STD)

    # Step 2: Perturb generator voltage setpoints
    voltage_perturbations = perturb_voltages!(data, VOLTAGE_NOISE_STD, VM_MIN, VM_MAX)

    # Step 3: Apply flat start
    apply_flat_start!(data)

    # Step 4: Solve power flow
    result, iterations = solve_pf_with_iterations(data)

    # Check convergence
    term_status = result["termination_status"]
    converged = (term_status == MOI.LOCALLY_SOLVED || term_status == MOI.OPTIMAL)

    if !converged
        n_failed += 1
        if n_failed <= 10
            println("  Sample $i: FAILED ($term_status)")
        end
        continue
    end

    n_converged += 1

    # Convert termination status to string (enums don't serialize to JSON)
    result["termination_status"] = string(result["termination_status"])

    # Build output structure — save everything
    sample_data = Dict(
        "sample_id" => i,
        "termination_status" => string(term_status),
        "solve_time_seconds" => result["solve_time"],
        "solver_iterations" => iterations,
        "flat_start" => true,
        "load_noise_std" => LOAD_NOISE_STD,
        "voltage_noise_std" => VOLTAGE_NOISE_STD,

        "data" => data,
        "result" => result,
    )

    # Save
    filename = "sample_$(lpad(i, 4, '0')).json"
    filepath = joinpath(OUTPUT_DIR, filename)
    save_sample(filepath, sample_data)

    # Progress
    if i % 100 == 0
        println("  Completed $i / $N_SAMPLES " *
                "($n_converged converged, $n_failed failed)")
    end
end
# --- Summary ---
println()
println("=" ^ 50)
println("Dataset generation complete!")
println("  Total attempted: $N_SAMPLES")
println("  Converged: $n_converged")
println("  Failed: $n_failed")
println("  Convergence rate: $(round(n_converged/N_SAMPLES*100, digits=1))%")
println("  Output directory: $OUTPUT_DIR")
println("=" ^ 50)

Loaded case: 57 buses, 7 generators, 42 loads, 80 branches

Starting dataset generation...
  Samples: 1000
  Load noise: 10.0%
  Voltage noise: 2.0%
  Voltage clamp: [0.9, 1.1] p.u.
  Flat start: enabled

  Completed 100 / 1000 (100 converged, 0 failed)
  Completed 200 / 1000 (200 converged, 0 failed)
  Completed 300 / 1000 (300 converged, 0 failed)
  Completed 400 / 1000 (400 converged, 0 failed)
  Completed 500 / 1000 (500 converged, 0 failed)
  Completed 600 / 1000 (600 converged, 0 failed)
  Completed 700 / 1000 (700 converged, 0 failed)
  Completed 800 / 1000 (800 converged, 0 failed)
  Completed 900 / 1000 (900 converged, 0 failed)
  Completed 1000 / 1000 (1000 converged, 0 failed)

Dataset generation complete!
  Total attempted: 1000
  Converged: 1000
  Failed: 0
  Convergence rate: 100.0%
  Output directory: pf_dataset


## Testing which V needs to be perturbed

In [1]:
using PowerModels
using Ipopt
using Suppressor

PowerModels.silence()

CASE_FILE = "pglib_opf_case57_ieee.m"

# --- Test 1: Set bus["vm"] to something extreme on a PV bus ---
println("=" ^ 50)
println("TEST 1: Extreme bus[\"vm\"] on a PV bus")
println("=" ^ 50)

data1 = PowerModels.parse_file(CASE_FILE)

# Find a PV bus (bus_type == 2)
pv_bus_id = nothing
for (bus_id, bus) in data1["bus"]
    if bus["bus_type"] == 2
        pv_bus_id = bus_id
        break
    end
end

println("PV bus ID: $pv_bus_id")
println("  Original bus[\"vm\"]: $(data1["bus"][pv_bus_id]["vm"])")

# Find the generator at this bus
gen_at_bus = nothing
for (gen_id, gen) in data1["gen"]
    if string(gen["gen_bus"]) == pv_bus_id || gen["gen_bus"] == parse(Int, pv_bus_id)
        gen_at_bus = gen_id
        break
    end
end

println("  Generator at this bus: $gen_at_bus")
println("  gen[\"vg\"]: $(data1["gen"][gen_at_bus]["vg"])")

# Set bus vm to something extreme
data1["bus"][pv_bus_id]["vm"] = 5.0
println("\n  Setting bus[\"vm\"] = 5.0 (extreme)")
println("  Keeping gen[\"vg\"] = $(data1["gen"][gen_at_bus]["vg"]) (unchanged)")

solver = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 5, "max_iter" => 100)

output1 = @capture_out begin
    global result1 = PowerModels.solve_pf(data1, ACPPowerModel, solver)
end

println("\n  Termination: $(result1["termination_status"])")
if haskey(result1["solution"], "bus")
    solved_vm = result1["solution"]["bus"][pv_bus_id]["vm"]
    println("  Solved bus vm: $solved_vm")
    println("  (Does it match gen vg=$(data1["gen"][gen_at_bus]["vg"]) or bus vm=5.0?)")
end

# --- Test 2: Set gen["vg"] to something extreme instead ---
println("\n" * "=" ^ 50)
println("TEST 2: Extreme gen[\"vg\"] on the same PV bus")
println("=" ^ 50)

data2 = PowerModels.parse_file(CASE_FILE)

println("  Original gen[\"vg\"]: $(data2["gen"][gen_at_bus]["vg"])")
println("  Original bus[\"vm\"]: $(data2["bus"][pv_bus_id]["vm"])")

# Set gen vg to something extreme, keep bus vm normal
data2["gen"][gen_at_bus]["vg"] = 5.0
println("\n  Setting gen[\"vg\"] = 5.0 (extreme)")
println("  Keeping bus[\"vm\"] = $(data2["bus"][pv_bus_id]["vm"]) (unchanged)")

output2 = @capture_out begin
    global result2 = PowerModels.solve_pf(data2, ACPPowerModel, solver)
end

println("\n  Termination: $(result2["termination_status"])")
if haskey(result2["solution"], "bus")
    solved_vm = result2["solution"]["bus"][pv_bus_id]["vm"]
    println("  Solved bus vm: $solved_vm")
    println("  (Does it match gen vg=5.0 or original bus vm?)")
end

# --- Test 3: Control — no changes ---
println("\n" * "=" ^ 50)
println("TEST 3: Control (no changes)")
println("=" ^ 50)

data3 = PowerModels.parse_file(CASE_FILE)
output3 = @capture_out begin
    global result3 = PowerModels.solve_pf(data3, ACPPowerModel, solver)
end

println("  Termination: $(result3["termination_status"])")
solved_vm = result3["solution"]["bus"][pv_bus_id]["vm"]
println("  Solved bus vm at PV bus: $solved_vm")
println("  gen[\"vg\"]: $(data3["gen"][gen_at_bus]["vg"])")
println("  bus[\"vm\"]: $(data3["bus"][pv_bus_id]["vm"])")

[info | PowerModels]: Suppressing information and warning messages for the rest of this session.  Use the Memento package for more fine-grained control of logging.
TEST 1: Extreme bus["vm"] on a PV bus
PV bus ID: 2
  Original bus["vm"]: 1.0
  Generator at this bus: 2
  gen["vg"]: 1.0

  Setting bus["vm"] = 5.0 (extreme)
  Keeping gen["vg"] = 1.0 (unchanged)

  Termination: LOCALLY_INFEASIBLE
  Solved bus vm: 4.99999999999917
  (Does it match gen vg=1.0 or bus vm=5.0?)

TEST 2: Extreme gen["vg"] on the same PV bus
  Original gen["vg"]: 1.0
  Original bus["vm"]: 1.0

  Setting gen["vg"] = 5.0 (extreme)
  Keeping bus["vm"] = 1.0 (unchanged)

  Termination: LOCALLY_SOLVED
  Solved bus vm: 1.0
  (Does it match gen vg=5.0 or original bus vm?)

TEST 3: Control (no changes)
  Termination: LOCALLY_SOLVED
  Solved bus vm at PV bus: 1.0
  gen["vg"]: 1.0
  bus["vm"]: 1.0


In [2]:
data = PowerModels.parse_file(CASE_FILE)

# Perturb bus["vm"] on PV bus slightly
pv_bus_id = "2"
data["bus"][pv_bus_id]["vm"] = 1.05
println("Set PV bus vm to 1.05")

solver = optimizer_with_attributes(Ipopt.Optimizer, "print_level" => 0)

output = @capture_out begin
    global result = PowerModels.solve_pf(data, ACPPowerModel, solver)
end

println("Termination: $(result["termination_status"])")
println("Solved vm at PV bus: $(result["solution"]["bus"][pv_bus_id]["vm"])")
# Should print ~1.05, confirming the solver respected our constraint

Set PV bus vm to 1.05
Termination: LOCALLY_SOLVED
Solved vm at PV bus: 1.05


In [ ]:
using PowerModels
using Ipopt
using JSON
using JuMP
using Random
using Suppressor

# Suppress PowerModels logging
PowerModels.silence()

# Set random seed for reproducibility
Random.seed!(42)

# --- Configuration ---
N_SAMPLES = 1000
LOAD_NOISE_STD = 0.1                     # 10% noise on loads
VOLTAGE_NOISE_STD = 0.02                 # 2% noise on voltage setpoints
VM_MIN = 0.9                             # Minimum allowable voltage setpoint
VM_MAX = 1.1                             # Maximum allowable voltage setpoint
CASE_FILE = "pglib_opf_case57_ieee.m"
OUTPUT_DIR = "pf_dataset"

mkpath(OUTPUT_DIR)

# --- Load base case ---
base_data = PowerModels.parse_file(CASE_FILE)
println("Loaded case: $(length(base_data["bus"])) buses, " *
        "$(length(base_data["gen"])) generators, " *
        "$(length(base_data["load"])) loads, " *
        "$(length(base_data["branch"])) branches")

# --- Helper: Apply flat start (CORRECTED) ---
"""
    apply_flat_start!(data)

Set initial guesses to flat values, WITHOUT changing PV/slack constraints.
Only modifies variables that the solver will actually solve for.
"""
function apply_flat_start!(data)
    for (bus_id, bus) in data["bus"]
        if bus["bus_type"] == 1  # PQ bus
            bus["vm"] = 1.0     # Initial guess for voltage magnitude
            bus["va"] = 0.0     # Initial guess for voltage angle
        elseif bus["bus_type"] == 2  # PV bus
            # Do NOT touch vm — it's the constraint!
            bus["va"] = 0.0     # Initial guess for voltage angle only
        elseif bus["bus_type"] == 3  # Slack bus
            # Do NOT touch vm or va — both are constraints!
        end
    end
end

# --- Helper: Perturb loads ---
"""
    perturb_loads!(data, noise_std)

Apply Gaussian noise to all active loads. Same multiplier
for pd and qd (preserves power factor).
"""
function perturb_loads!(data, noise_std)
    multipliers = Dict{String, Float64}()

    for (load_id, load) in data["load"]
        if load["status"] == 0
            continue
        end

        if load["pd"] == 0.0 && load["qd"] == 0.0
            multipliers[load_id] = 1.0
            continue
        end

        multiplier = 1.0 + noise_std * randn()
        load["pd"] *= multiplier
        load["qd"] *= multiplier
        multipliers[load_id] = multiplier
    end

    return multipliers
end

# --- Helper: Perturb voltage setpoints (CORRECTED) ---
"""
    perturb_voltages!(data, noise_std, vm_min, vm_max)

Perturb bus["vm"] on PV and slack buses — these are the actual
voltage setpoint constraints used by solve_pf.
"""
function perturb_voltages!(data, noise_std, vm_min, vm_max)
    voltage_perturbations = Dict{String, Dict{String, Any}}()

    for (bus_id, bus) in data["bus"]
        # Only perturb PV (type 2) and slack (type 3) buses
        if bus["bus_type"] == 1  # PQ bus — vm is solved, not a setpoint
            continue
        end

        original_vm = bus["vm"]

        # Add noise and clamp
        perturbed_vm = original_vm * (1.0 + noise_std * randn())
        perturbed_vm = clamp(perturbed_vm, vm_min, vm_max)

        bus["vm"] = perturbed_vm

        voltage_perturbations[bus_id] = Dict(
            "bus_type" => bus["bus_type"],
            "original_vm" => original_vm,
            "perturbed_vm" => perturbed_vm
        )
    end

    return voltage_perturbations
end

# --- Helper: Solve power flow with iteration count ---
function solve_pf_with_iterations(data)
    solver = optimizer_with_attributes(
        Ipopt.Optimizer,
        "print_level" => 5,
        "max_iter" => 100,
        "tol" => 1e-8,
    )

    local result
    output = @capture_out begin
        result = PowerModels.solve_pf(data, ACPPowerModel, solver)
    end

    # Parse iteration count from Ipopt output
    iterations = -1
    for line in split(output, "\n")
        if contains(line, "Number of Iterations")
            iterations = parse(Int, strip(split(line, ":")[end]))
            break
        end
    end

    return result, iterations
end

# --- Helper: Save sample ---
function save_sample(filepath, sample_data)
    open(filepath, "w") do f
        JSON.print(f, sample_data, 4)
    end
end

# --- Main generation loop ---
println("\nStarting dataset generation...")
println("  Samples: $N_SAMPLES")
println("  Load noise: $(LOAD_NOISE_STD * 100)%")
println("  Voltage noise: $(VOLTAGE_NOISE_STD * 100)%")
println("  Voltage clamp: [$VM_MIN, $VM_MAX] p.u.")
println("  Flat start: enabled")
println()

n_converged = 0
n_failed = 0

for i in 1:N_SAMPLES
    # Fresh copy of base case
    data = deepcopy(base_data)

    # Step 1: Perturb loads
    load_multipliers = perturb_loads!(data, LOAD_NOISE_STD)

    # Step 2: Perturb generator voltage setpoints
    voltage_perturbations = perturb_voltages!(data, VOLTAGE_NOISE_STD, VM_MIN, VM_MAX)

    # Step 3: Apply flat start
    apply_flat_start!(data)

    # Step 4: Solve power flow
    result, iterations = solve_pf_with_iterations(data)

    # Check convergence
    term_status = result["termination_status"]
    converged = (term_status == MOI.LOCALLY_SOLVED || term_status == MOI.OPTIMAL)

    if !converged
        n_failed += 1
        if n_failed <= 10
            println("  Sample $i: FAILED ($term_status)")
        end
        continue
    end

    n_converged += 1

    # Convert termination status to string (enums don't serialize to JSON)
    result["termination_status"] = string(result["termination_status"])

    # Build output structure — save everything
    sample_data = Dict(
        "sample_id" => i,
        "termination_status" => string(term_status),
        "solve_time_seconds" => result["solve_time"],
        "solver_iterations" => iterations,
        "flat_start" => true,
        "load_noise_std" => LOAD_NOISE_STD,
        "voltage_noise_std" => VOLTAGE_NOISE_STD,

        "data" => data,
        "result" => result,
    )
    
    # Save
    filename = "sample_$(lpad(i, 4, '0')).json"
    filepath = joinpath(OUTPUT_DIR, filename)
    save_sample(filepath, sample_data)

    # Progress
    if i % 100 == 0
        println("  Completed $i / $N_SAMPLES " *
                "($n_converged converged, $n_failed failed)")
    end
end
# --- Summary ---
println()
println("=" ^ 50)
println("Dataset generation complete!")
println("  Total attempted: $N_SAMPLES")
println("  Converged: $n_converged")
println("  Failed: $n_failed")
println("  Convergence rate: $(round(n_converged/N_SAMPLES*100, digits=1))%")
println("  Output directory: $OUTPUT_DIR")
println("=" ^ 50)